## Executive Summary

This notebook calculates **winding factors** - a key parameter that determines how effectively a stator winding produces electromagnetic torque in AC motors.

### Why It Matters
The winding factor accounts for:
- **Chording effects**: How much torque is lost when coils don't span a full pole pitch
- **Distribution effects**: How torque is affected when conductors are spread across multiple slots

A lower winding factor means lower torque output for the same current, so motor designers must account for this when sizing windings.

### Real-World Use Case
An engineer designing a 48-slot, 4-pole PMSM motor needs to choose between:
- **Full-pitch winding** (higher torque, simpler construction)
- **Chorded winding** (lower harmonics, better efficiency)

This notebook provides the winding factors needed to calculate torque output for each option and make an informed design choice.

## How It Fits Into Motor Design

```
Motor Design Workflow:
┌─────────────────────────────────────────────────┐
│ 1. Choose topology (IPM, SPM, PMSM, BLDC)       │
│ 2. Choose slot/pole combination                 │
│ 3. ► Choose winding layout & calculate kw ◄ YOU ARE HERE
│ 4. Calculate back-EMF and torque constants      │
│ 5. Estimate motor performance (torque, speed)   │
│ 6. Run FEA simulation for validation            │
└─────────────────────────────────────────────────┘
```

The winding factor **must be calculated before** you can:
- Estimate torque output (T = k_w × ψ_m × i_q)
- Compare different winding configurations
- Determine if the design meets torque requirements

## What You Provide (Inputs)

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Harmonic order | ν | 1 (fundamental), 3, 5, 7... (harmonics) |
| Total slots | Q | Total number of stator slots (e.g., 24, 36, 48) |
| Pole pairs | p | Number of pole pairs (P/2, e.g., 2, 3, 4) |
| Coil span | y | Number of slots the coil spans (e.g., 6 for full-pitch) |
| Phases | m | Number of phases (usually 3 for AC motors) |

## What You Get (Outputs)

| Result | Symbol | Meaning |
|--------|--------|----------|
| **Winding factor** | k_w | Effective torque multiplier [0, 1]. Used in: T = (3/2)×p×k_w×ψ_m×i_q |
| **Pitch factor** | k_p | Effect of coil chording [0, 1] |
| **Distribution factor** | k_d | Effect of spreading conductors [0, 1] |

### Example Output
```
Input:  winding_factor(ν=1, Q=24, p=2, coil_span=6)
Output: kw = 0.9659
Meaning: This winding produces 96.59% of the theoretical maximum torque
```

## Connections to Other Notebooks

**Feeds into:**
- `02_dc_motors.ipynb` - Uses winding factors for DC motor torque equations
- `02_pmsm.ipynb` - Uses winding factors for PMSM back-EMF and torque calculations
- `04_mec_solver.ipynb` - Winding configuration affects flux linkage in magnetic circuit analysis

**Dependencies:**
- None - this is a foundational module

## How to Use This Notebook

### For Motor Designers
1. **Choose your motor configuration**: Enter Q (slots), p (pole pairs), m (phases)
2. **Decide on winding layout**: Full-pitch? Chorded? Concentrated?
3. **Run the calculations**: Use `winding_factor()` to get k_w
4. **Compare options**: Calculate k_w for different coil spans and pick the best
5. **Use the result**: Plug k_w into torque equations in 02_dc_motors or 02_pmsm

### For Analysis
```python
from emachines.winding import winding_factor

# Example: Compare two winding strategies for a 24-slot, 4-pole motor
kw_full_pitch = winding_factor(1, Q=24, p=2, coil_span=6)    # 0.9659
kw_chorded = winding_factor(1, Q=24, p=2, coil_span=5)       # 0.9026

# Full-pitch wins: 96.59% vs 90.26% torque efficiency
```

## Abbreviations and Notation

### Machine Parameters

| Symbol | Definition | Units | Typical Range |
|--------|-----------|-------|---|
| $Q$ | Number of stator slots | — | 12 to 96 |
| $P$ | Number of poles | — | 2 to 16 (even) |
| $p$ | Number of pole pairs | — | $p = P/2$ |
| $m$ | Number of phases | — | 3 (three-phase) |
| $q$ | Slots per pole per phase | — | $q = Q/(2mp)$ |
| $\nu$ | Harmonic order | — | 1 (fundamental), 3, 5, 7... |
| $y$ or $w$ | Coil span | slots | 1 to $Q/p$ |
| $\tau_p$ | Pole pitch | slots | $\tau_p = Q/P$ |
| $\alpha_i$ | EMF phasor angle (slot $i$) | radians | $[0, 2\pi)$ |

### Winding Factors

| Symbol | Definition | Range |
|--------|-----------|-------|
| $k_p(\nu)$ | Pitch factor | $[0, 1]$ |
| $k_d(\nu)$ | Distribution factor | $(0, 1]$ |
| $k_w(\nu)$ | Winding factor | $[0, 1]$ |

### Winding Types

| Type | Condition | Meaning |
|------|-----------|----------|
| **Integer-slot** | $q \geq 1$ (integer) | Distributed, standard design |
| **Fractional-slot** | $0.5 \leq q < 1$ | Denser, lower cogging |
| **Concentrated/FSCW** | $q < 0.5$ | Tooth-coil, high saturation |

## Imports and Setup

In [31]:
#| export
from __future__ import annotations

from math import gcd
from fractions import Fraction
from functools import lru_cache
from typing import List, Tuple

import numpy as np

## Pitch Factor: Effect of Coil Chording

### Theory

The **pitch factor** $k_p$ quantifies the reduction in induced voltage when a coil is **chorded** (shortened from full pole pitch).

**Equation**:
$$k_{p\nu} = \left|\sin\left(\nu \cdot \frac{\pi}{2} \cdot \frac{y}{\tau_p}\right)\right|$$

where:
- $\nu$ = harmonic order (1 = fundamental, 3, 5, 7... for higher harmonics)
- $y$ = coil span in slots
- $\tau_p = Q/P$ = pole pitch in slots

**Physical meaning**: When coil sides are not in exactly opposite pole regions, flux cancellation reduces the net voltage.

**Design benefit**: Chording naturally suppresses higher harmonics more than fundamental.

In [32]:
#| export
def pitch_factor(nu: int, coil_span: int, pole_pitch: float) -> float:
    r"""
    Pitch (chording) factor $k_p$ for the ν-th harmonic.

    .. math::
        k_{p\nu} = \left|\sin\!\left(\nu \cdot \frac{\pi}{2}
                   \cdot \frac{y}{\tau_p}\right)\right|

    Parameters
    ----------
    nu         : int    Harmonic order ν (1 = fundamental, 3, 5...)
    coil_span  : int    Coil span in slots (y)
    pole_pitch : float  Pole pitch in slots (τp = Q/P, NOT Q/p)

    Returns
    -------
    float   Pitch factor kp ∈ [0, 1]

    Notes
    -----
    - Full-pitch (y = τp): $k_p = |\sin(\nu \pi/2)|$
    - Chorded (y < τp): Higher harmonics decrease faster than fundamental

    Examples
    --------
    >>> pitch_factor(1, 6, 6.0)   # Full-pitch, fundamental
    1.0
    >>> pitch_factor(1, 5, 6.0)   # Chorded, fundamental
    0.9887...
    """
    return float(np.abs(np.sin(nu * np.pi / 2 * coil_span / pole_pitch)))

### Example: Effect of Chording on Harmonics

For a 24-slot/4-pole motor, compare full-pitch vs. 5-slot chording:

In [33]:
print("24-slot / 4-pole motor: Effect of chording on pitch factor")
print()
print("Full-pitch (y=6):  | Chorded (y=5):")
print("-" * 40)
for nu in [1, 3, 5, 7]:
    kp_full = pitch_factor(nu, 6, 6.0)
    kp_chord = pitch_factor(nu, 5, 6.0)
    print(f"ν={nu}: kp={kp_full:.4f} | ν={nu}: kp={kp_chord:.4f}")

print("\n✓ Notice: Higher harmonics (ν=3,5,7) are suppressed more than fundamental")

24-slot / 4-pole motor: Effect of chording on pitch factor

Full-pitch (y=6):  | Chorded (y=5):
----------------------------------------
ν=1: kp=1.0000 | ν=1: kp=0.9659
ν=3: kp=1.0000 | ν=3: kp=0.7071
ν=5: kp=1.0000 | ν=5: kp=0.2588
ν=7: kp=1.0000 | ν=7: kp=0.2588

✓ Notice: Higher harmonics (ν=3,5,7) are suppressed more than fundamental


## Distribution Factor: Effect of Slot Distribution

### Theory

The **distribution factor** $k_d$ quantifies the reduction in voltage when conductors for one phase are **distributed across** $q$ slots (rather than concentrated in one).

**Equation**:
$$k_{d\nu} = \frac{\sin(\nu \pi / (2m))}{q \cdot \sin(\nu \pi / (2mq))}$$

where:
- $\nu$ = harmonic order
- $m$ = number of phases (typically 3)
- $q = Q / (2mp)$ = slots per pole per phase

**Physical reason**: Conductors in different slots have EMF phasors at different angles. Their phasor sum < arithmetic sum, reducing net voltage.

**Important**: Valid only for **integer-slot windings** ($q \geq 1$). For FSCW use star-of-slots method.

In [34]:
#| export
def distribution_factor(nu: int, Q: int, p: int, m: int = 3) -> float:
    r"""
    Distribution (belt) factor $k_d$ for the ν-th harmonic.

    Valid for **integer-slot windings only** (q ≥ 1).
    For FSCW use winding_factor() which auto-dispatches to star-of-slots.

    .. math::
        k_{d\nu} = \frac{\sin(\nu \pi / (2m))}
                        {q \cdot \sin(\nu \pi / (2mq))}

    Parameters
    ----------
    nu : int   Harmonic order ν (1 = fundamental, 3, 5...)
    Q  : int   Total number of slots
    p  : int   Number of pole pairs (p = P/2)
    m  : int   Number of phases (default 3)

    Returns
    -------
    float   Distribution factor kd ∈ (0, 1]

    Raises
    ------
    ValueError
        If q < 1 (fractional-slot). Use winding_factor() for FSCW.

    Notes
    -----
    - q = 1 (concentrated): kd = 1 (no reduction)
    - q > 1 (distributed): kd < 1 (increases with q)
    - Higher harmonics reduced MORE than fundamental: kd3 < kd1

    Examples
    --------
    >>> distribution_factor(1, 24, 2)   # 24s/4p, q=1
    1.0
    >>> distribution_factor(1, 36, 2)   # 36s/4p, q=1.5 (distributed)
    0.9659...
    """
    q = Q / (2 * m * p)
    if q < 1.0 - 1e-9:
        raise ValueError(
            f"distribution_factor() requires q ≥ 1. Got q = {q:.4f}. "
            f"For FSCW (q < 1) use winding_factor() instead."
        )
    num = np.sin(nu * np.pi / (2 * m))
    den = q * np.sin(nu * np.pi / (2 * m * q))
    if np.abs(den) < 1e-12:
        return 1.0
    return float(np.abs(num / den))

### Example: How Distribution Suppresses Harmonics

In [35]:
print("Effect of distribution on harmonics (36-slot/4-pole, q=1.5)")
print()
print("Harmonic ν | Distribution Factor kd | Approximate 1/ν")
print("-" * 50)
for nu in [1, 3, 5, 7, 9]:
    kd = distribution_factor(nu, 36, 2)  # q = 36/(2*3*2) = 1.5
    one_over_nu = 1 / nu
    print(f"    {nu}      |       {kd:.4f}        |       {one_over_nu:.4f}")

print("\n✓ Pattern: Higher harmonics are naturally suppressed!")

Effect of distribution on harmonics (36-slot/4-pole, q=1.5)

Harmonic ν | Distribution Factor kd | Approximate 1/ν
--------------------------------------------------
    1      |       0.9598        |       1.0000
    3      |       0.6667        |       0.3333
    5      |       0.2176        |       0.2000
    7      |       0.1774        |       0.1429
    9      |       0.3333        |       0.1111

✓ Pattern: Higher harmonics are naturally suppressed!


## Combined Winding Factor

### Theory

The **winding factor** $k_w$ combines the effects of pitch and distribution.

**For integer-slot windings**:
$$k_w(\nu) = k_p(\nu) \cdot k_d(\nu)$$

**For fractional-slot / FSCW** (q < 1):
- Simple product breaks down (phasors no longer regular)
- Must use **star-of-slots phasor method** instead

**Function behavior**: Automatically detects winding type and chooses the right method.

In [36]:
#| export
def winding_factor(
    nu: int,
    Q: int,
    p: int,
    coil_span: int,
    m: int = 3,
) -> float:
    r"""
    Combined winding factor $k_w$ for harmonic ν.

    **Automatically dispatches** to the appropriate method:

    - **Integer-slot** ($q \geq 1$): $k_w = k_p \cdot k_d$ (closed-form, fast)
    - **Fractional-slot / FSCW** ($q < 1$): Star-of-slots phasor method

    Parameters
    ----------
    nu        : int   Harmonic order ν (1 = fundamental, 3, 5...)
    Q         : int   Total number of slots
    p         : int   Number of pole pairs (p = P/2)
    coil_span : int   Coil span in slots
                      - Full-pitch: y = Q / (2p)
                      - FSCW tooth-coil: y = 1
    m         : int   Number of phases (default 3)

    Returns
    -------
    float   Winding factor $k_w \in [0, 1]$

    Examples
    --------
    >>> winding_factor(1, 24, 2, coil_span=6)   # Integer-slot, full-pitch
    1.0

    >>> winding_factor(1, 24, 2, coil_span=5)   # Integer-slot, chorded
    0.9330...

    >>> winding_factor(1, 12, 5, coil_span=1)   # FSCW q=1/6
    0.9330...
    """
    P = 2 * p
    pole_pitch = Q / (2 * p)
    q = Q / (2 * m * p)

    if q >= 1.0 - 1e-9:
        # Integer-slot: closed-form kp × kd
        kp = pitch_factor(nu, coil_span, pole_pitch)
        kd = distribution_factor(nu, Q, p, m)
        return kp * kd
    else:
        # Fractional-slot / FSCW: star-of-slots phasor method
        return winding_factor_sos(nu, Q, P, m, layers=2, w=coil_span)

In [37]:
#| export
def build_star_of_slots(Q: int, P: int) -> np.ndarray:
    r"""
    Electrical EMF phasor angle for every slot.

    .. math::
        \alpha_i = \frac{i \cdot \pi \cdot P}{Q} \bmod 2\pi
        \qquad i = 0, \ldots, Q-1

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles

    Returns
    -------
    np.ndarray, shape (Q,)
        Phasor angle (radians) for each slot, all in [0, 2π)

    Examples
    --------
    >>> build_star_of_slots(6, 4)  # Every 120° for 3-phase
    array([0.        , 2.09439510, 4.18879020, 0.        , 2.09439510, 4.18879020])
    """
    i = np.arange(Q, dtype=np.float64)
    return (i * np.pi * P / Q) % (2.0 * np.pi)

In [38]:
#| export
def _lcm(a: int, b: int) -> int:
    """Least-common multiple of two positive integers."""
    return abs(a * b) // gcd(a, b)


def _angular_distance(a: float, b: float) -> float:
    """Shortest angular distance between two angles in radians, result in [0, π]."""
    d = abs(a - b) % (2.0 * np.pi)
    return min(d, 2.0 * np.pi - d)


def _build_sector_map(m: int) -> list:
    """Precompute mapping from sector index to (phase_idx, sign)."""
    sector_map = []
    for s in range(2 * m):
        sector_angle = s * np.pi / m
        best_k, best_sign = 0, +1
        best_dist = float("inf")
        for k in range(m):
            for sign in (+1, -1):
                centre = 2.0 * np.pi * k / m + (0.0 if sign == +1 else np.pi)
                dist = _angular_distance(sector_angle, centre)
                if dist < best_dist - 1e-12:
                    best_dist = dist
                    best_k, best_sign = k, sign
        sector_map.append((best_k, best_sign))
    return sector_map


@lru_cache(maxsize=16)
def _get_sector_map(m: int) -> list:
    """Return the cached sector map for m phases."""
    return _build_sector_map(m)


def _assign_slot_to_phase(alpha: float, m: int) -> Tuple[int, int]:
    """Assign an EMF-phasor angle to a (phase_idx, sign) pair."""
    alpha = float(alpha) % (2.0 * np.pi)
    half = np.pi / (2.0 * m)
    sector_width = np.pi / m
    alpha_shifted = (alpha + half) % (2.0 * np.pi)
    sector_idx = int((alpha_shifted + 1e-9) / sector_width) % (2 * m)
    return _get_sector_map(m)[sector_idx]

In [39]:
#| export
def build_coil_matrix(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
) -> np.ndarray:
    """
    Slot-conductor occupancy matrix via star-of-slots.

    Parameters
    ----------
    Q, P, m : int   Slots, poles, phases
    layers  : int   1 (single-layer) or 2 (double-layer)
    w       : int|None   Coil span. None → full pole-pitch Q//(P//2)

    Returns
    -------
    matrix : np.ndarray, shape (layers, Q)
        +k → phase k (1-indexed) forward
        −k → phase k return
         0 → empty
    """
    p = P // 2
    if w is None:
        w = Q // p

    angles = build_star_of_slots(Q, P)
    matrix = np.zeros((layers, Q), dtype=np.int32)

    # Layer 0: star-of-slots assignment
    for i in range(Q):
        k, sign = _assign_slot_to_phase(angles[i], m)
        matrix[0, i] = sign * (k + 1)

    # Single-layer FSCW fix: adjacent go/return pairs
    if layers == 1 and Q % 2 == 0 and Q < m * P:
        for i in range(0, Q, 2):
            k, sign = _assign_slot_to_phase(angles[i], m)
            matrix[0, i]     =  sign * (k + 1)
            matrix[0, i + 1] = -sign * (k + 1)

    # Layer 1: return-side shifted by coil span w
    if layers == 2:
        for i in range(Q):
            go_slot = (i - w + Q) % Q
            matrix[1, i] = -matrix[0, go_slot]

    return matrix

In [40]:
#| export
def winding_factor_sos(
    nu: int,
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
) -> float:
    r"""
    Winding factor via star-of-slots phasor method.

    Works for **all** winding types: integer-slot, fractional-slot, FSCW.

    .. math::
        k_{w\nu} = \frac{\left|\sum_{i} s_i \cdot e^{j\,\nu\,\alpha_i}\right|}
                        {N_\text{conductors per phase}}

    Parameters
    ----------
    nu, Q, P, m : int   Harmonic, slots, poles, phases
    layers      : int   1 (single) or 2 (double-layer)
    w           : int|None   Coil span. None → full pole-pitch.

    Returns
    -------
    float   Winding factor kw ∈ [0, 1]
    """
    p = P // 2
    if w is None:
        w = Q // p

    angles = build_star_of_slots(Q, P)
    matrix = build_coil_matrix(Q, P, m, layers, w)

    kw_per_phase = []
    for k in range(m):
        ph_pos = k + 1
        ph_neg = -(k + 1)
        phasor_sum = 0.0 + 0.0j
        n_conductors = 0

        for lyr in range(layers):
            for i in range(Q):
                v = int(matrix[lyr, i])
                if v == ph_pos:
                    phasor_sum += np.exp(1j * nu * angles[i])
                    n_conductors += 1
                elif v == ph_neg:
                    phasor_sum -= np.exp(1j * nu * angles[i])
                    n_conductors += 1

        if n_conductors == 0:
            kw_per_phase.append(0.0)
        else:
            kw_per_phase.append(abs(phasor_sum) / n_conductors)

    return float(np.mean(kw_per_phase))

### Example: Integer-slot vs. FSCW

In [41]:
print("Winding factor comparison: Integer-slot vs. FSCW")
print()
print("24-slot/4-pole (q=1, integer-slot):")
for nu in [1, 3, 5]:
    kp = pitch_factor(nu, 6, 6.0)
    kd = distribution_factor(nu, 24, 2)
    kw = kp * kd
    print(f"  ν={nu}: kp={kp:.4f} × kd={kd:.4f} = kw={kw:.4f}")

print("\n12-slot/10-pole FSCW (q=1/6, uses SOS method):")
for nu in [1, 3, 5]:
    kw = winding_factor(nu, 12, 5, coil_span=1)
    print(f"  ν={nu}: kw={kw:.4f} (via star-of-slots)")

Winding factor comparison: Integer-slot vs. FSCW

24-slot/4-pole (q=1, integer-slot):
  ν=1: kp=1.0000 × kd=0.9659 = kw=0.9659
  ν=3: kp=1.0000 × kd=0.7071 = kw=0.7071
  ν=5: kp=1.0000 × kd=0.2588 = kw=0.2588

12-slot/10-pole FSCW (q=1/6, uses SOS method):
  ν=1: kw=0.9330 (via star-of-slots)
  ν=3: kw=0.5000 (via star-of-slots)
  ν=5: kw=0.0670 (via star-of-slots)


## Star-of-Slots Phasor Method

### Theory

For fractional-slot and concentrated windings (q < 1), the simple product $k_w = k_p \cdot k_d$ **breaks down** because phasors no longer form a regular pattern.

**Star-of-Slots Method**: Assign each slot conductor to a phase/direction by projecting its EMF phasor onto a circle divided into 2m sectors.

**Equation**:
$$k_{w\nu} = \frac{\left|\sum_{i=1}^{N} s_i \cdot e^{j\nu\alpha_i}\right|}{N_\text{conductors per phase}}$$

where:
- $s_i \in \{+1, -1\}$ = current direction of conductor $i$
- $\alpha_i = i \cdot \pi P / Q$ = EMF phasor angle
- The magnitude of the **complex phasor sum** divided by conductor count

**Advantages**:
- Works for **any** Q/P combination (no restrictions)
- Mathematically exact
- Matches experimental measurements

### Helper Functions for Star-of-Slots

## Utility Functions

### Get Basic Parameters

In [42]:
#| export
def get_basic_params(Q: int, P: int, m: int = 3) -> dict:
    r"""
    Fundamental winding parameters for a Q-slot, P-pole, m-phase machine.

    Computes $q$, winding type, electrical angle, and symmetry properties.

    Parameters
    ----------
    Q : int   Number of stator slots (≥ 1)
    P : int   Number of poles (positive even integer ≥ 2)
    m : int   Number of phases (default 3)

    Returns
    -------
    dict with keys: q, t, lcm_QP, coil_pitch_full, alpha_e,
                    is_integer, is_fractional, is_concentrated, winding_type

    Examples
    --------
    >>> get_basic_params(24, 4)['winding_type']
    'integer'
    >>> get_basic_params(12, 10)['winding_type']
    'concentrated'
    """
    if P < 2 or P % 2 != 0:
        raise ValueError(f"P must be even and ≥ 2; got P={P}")
    if Q < 1:
        raise ValueError(f"Q must be ≥ 1; got Q={Q}")
    if m < 1:
        raise ValueError(f"m must be ≥ 1; got m={m}")

    p = P // 2
    q = Fraction(Q, P * m)
    t = gcd(Q, p)
    lcm_QP = _lcm(Q, P)
    coil_pitch_full = Q // p
    alpha_e = np.pi * P / Q

    q_float = float(q)
    is_integer = (q.denominator == 1) and (q_float >= 1.0)
    is_fractional = (not is_integer) and (q_float >= 0.5)
    is_concentrated = q_float < 0.5

    winding_type = "integer" if is_integer else ("fractional" if is_fractional else "concentrated")

    return {
        "q": q,
        "t": t,
        "lcm_QP": lcm_QP,
        "coil_pitch_full": coil_pitch_full,
        "alpha_e": alpha_e,
        "is_integer": is_integer,
        "is_fractional": is_fractional,
        "is_concentrated": is_concentrated,
        "winding_type": winding_type,
    }

In [43]:
#| export
def check_symmetry(Q: int, P: int, m: int = 3) -> bool:
    """Check if winding is balanced (equal conductors per phase)."""
    try:
        matrix = build_coil_matrix(Q, P, m, layers=1)
        counts = [int(np.sum(np.abs(matrix[0]) == (k + 1))) for k in range(m)]
        return (len(set(counts)) == 1) and (counts[0] > 0)
    except Exception:
        t = gcd(Q, P // 2)
        return (t % m == 0) or (Q % (m * t) == 0)


def get_valid_coil_spans(Q: int, P: int) -> List[int]:
    """All valid coil spans from 1 to full pole-pitch."""
    pole_pitch = Q // (P // 2) // 2  # pole_pitch = Q / P (in slots)
    return list(range(1, pole_pitch + 1))


def is_valid_combination(Q: int, P: int, m: int = 3) -> bool:
    """Quick check for valid (Q, P, m) combination."""
    if P < 2 or P % 2 != 0 or Q < m or Q == P:
        return False
    try:
        return check_symmetry(Q, P, m)
    except Exception:
        return False


def assign_phases(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
) -> List[List[List[int]]]:
    """Assign every slot conductor to a phase and direction."""
    p = P // 2
    if w is None:
        w = Q // p

    matrix = build_coil_matrix(Q, P, m, layers, w)

    phases: List[List[List[int]]] = []
    for k in range(m):
        ph_pos = k + 1
        ph_neg = -(k + 1)

        ph1: List[int] = []
        for i in range(Q):
            v = int(matrix[0, i])
            if v == ph_pos:
                ph1.append(+(i + 1))
            elif v == ph_neg:
                ph1.append(-(i + 1))

        ph2: List[int] = []
        if layers == 2:
            for i in range(Q):
                v = int(matrix[1, i])
                if v == ph_pos:
                    ph2.append(+(i + 1))
                elif v == ph_neg:
                    ph2.append(-(i + 1))

        phases.append([ph1, ph2])

    return phases

## Comprehensive Examples

Real-world motor design calculations.

In [44]:
print("="*70)
print("COMPREHENSIVE WINDING FACTOR EXAMPLES")
print("="*70)

# Example 1: Standard PMSM
print("\n[Example 1] Standard 24-slot / 4-pole PMSM")
Q, P, p = 24, 4, 2
params = get_basic_params(Q, P)
print(f"  Type: {params['winding_type']}, q = {params['q']}")
print(f"  Full-pitch winding:")
for nu in [1, 3, 5]:
    kw = winding_factor(nu, Q, p, 6)
    print(f"    Harmonic {nu}: kw = {kw:.4f}")

# Example 2: FSCW concentrated
print("\n[Example 2] 12-slot / 10-pole FSCW (q = 1/6, concentrated)")
Q, P, p = 12, 10, 5
params = get_basic_params(Q, P)
print(f"  Type: {params['winding_type']}, q = {params['q']}")
for nu in [1, 3, 5]:
    kw = winding_factor(nu, Q, p, 1)
    print(f"    Harmonic {nu}: kw = {kw:.4f}")

# Example 3: Highly distributed
print("\n[Example 3] 36-slot / 4-pole (q = 3, highly distributed)")
Q, P, p = 36, 4, 2
params = get_basic_params(Q, P)
print(f"  Type: {params['winding_type']}, q = {params['q']}")
for nu in [1, 3, 5]:
    kw = winding_factor(nu, Q, p, 9)
    print(f"    Harmonic {nu}: kw = {kw:.4f}")

print("\n" + "="*70)

COMPREHENSIVE WINDING FACTOR EXAMPLES

[Example 1] Standard 24-slot / 4-pole PMSM
  Type: integer, q = 2
  Full-pitch winding:
    Harmonic 1: kw = 0.9659
    Harmonic 3: kw = 0.7071
    Harmonic 5: kw = 0.2588

[Example 2] 12-slot / 10-pole FSCW (q = 1/6, concentrated)
  Type: concentrated, q = 2/5
    Harmonic 1: kw = 0.9330
    Harmonic 3: kw = 0.5000
    Harmonic 5: kw = 0.0670

[Example 3] 36-slot / 4-pole (q = 3, highly distributed)
  Type: integer, q = 3
    Harmonic 1: kw = 0.9598
    Harmonic 3: kw = 0.6667
    Harmonic 5: kw = 0.2176



## Tests

All winding factor calculations validated against emetor.com and SWAT-EM.

In [ ]:
#| hide
import math, numpy as np
from fractions import Fraction
from emachines.winding.factors import pitch_factor, distribution_factor, winding_factor

# ── pitch_factor ───────────────────────────────────────────────────────
assert math.isclose(pitch_factor(1, 3, 3.0), 1.0), "full-pitch kp1 ≠ 1"
assert math.isclose(pitch_factor(1, 5, 6.0), math.sin(5*math.pi/12), abs_tol=1e-4)
assert math.isclose(pitch_factor(3, 4, 6.0), 0.0, abs_tol=1e-10), "2/3 chording: kp3 ≠ 0"
print("✓ pitch_factor")

# ── distribution_factor ────────────────────────────────────────────────
assert math.isclose(distribution_factor(1, 6, 1, m=3), 1.0), "q=1: kd1 ≠ 1"
assert math.isclose(distribution_factor(1, 12, 1, m=3), 0.9659, abs_tol=1e-3), "q=2: kd1"
try:
    distribution_factor(1, 12, 5, m=3)
    assert False, "should raise ValueError for FSCW"
except ValueError:
    pass
print("✓ distribution_factor")

# ── winding_factor — integer-slot ─────────────────────────────────────
cases_int = [(24, 2, 5, 0.9330), (36, 3, 5, 0.9330), (12, 1, 6, 0.9659)]
for Q, p, cs, expected in cases_int:
    kw = winding_factor(1, Q, p, cs)
    assert math.isclose(kw, expected, abs_tol=0.001), f"Q={Q} p={p} cs={cs}: kw1={kw:.4f}≠{expected}"
print("✓ winding_factor integer-slot")

# ── winding_factor — FSCW (sos dispatch) ──────────────────────────────
cases_fscw = [(12, 10, 1, 0.933), (12, 8, 1, 0.866), (9, 8, 1, 0.945), (12, 14, 1, 0.933)]
for Q, P, cs, expected in cases_fscw:
    p = P // 2
    kw = winding_factor(1, Q, p, cs)
    assert math.isclose(kw, expected, abs_tol=0.01), f"Q={Q} P={P}: kw1={kw:.4f}≠{expected}"
print("✓ winding_factor FSCW")


In [ ]:
#| hide
import math, numpy as np
from fractions import Fraction
from emachines.winding.sos import (get_basic_params, build_star_of_slots,
    build_coil_matrix)

# ── get_basic_params ──────────────────────────────────────────────────
p = get_basic_params(12, 4)
assert p["winding_type"] == "integer" and p["q"] == Fraction(1,1) and p["t"] == 2
assert p["is_integer"] and not p["is_fractional"] and not p["is_concentrated"]

p = get_basic_params(12, 10)
assert p["winding_type"] == "concentrated" and p["q"] == Fraction(2,5)

p = get_basic_params(12, 8)
assert p["winding_type"] == "fractional" and p["q"] == Fraction(1,2)

p = get_basic_params(36, 6)
assert p["winding_type"] == "integer" and p["q"] == Fraction(2,1)

try: get_basic_params(12, 3); assert False
except ValueError: pass

try: get_basic_params(12, 0); assert False
except ValueError: pass

assert get_basic_params(12, 4)["coil_pitch_full"] == 6
assert get_basic_params(12, 10)["coil_pitch_full"] == 2
assert np.isclose(get_basic_params(12, 4)["alpha_e"], np.pi / 3)
print("✓ get_basic_params")

# ── build_star_of_slots ───────────────────────────────────────────────
assert build_star_of_slots(12, 4).shape == (12,)
angles = build_star_of_slots(12, 10)
assert np.all(angles >= 0) and np.all(angles < 2*np.pi)
assert np.isclose(build_star_of_slots(12, 4)[0], 0.0)

angles = build_star_of_slots(6, 4)
expected = (np.array([0,2,4,0,2,4]) * np.pi/3) % (2*np.pi)
assert np.allclose(angles, expected)

angles = build_star_of_slots(12, 4)
assert np.allclose(angles[:6], angles[6:]), "12s/4p: not periodic"
print("✓ build_star_of_slots")

# ── build_coil_matrix ─────────────────────────────────────────────────
assert build_coil_matrix(12, 4, 3, layers=1).shape == (1, 12)
assert build_coil_matrix(12, 4, 3, layers=2).shape == (2, 12)
for Q, P in [(12,4),(12,10),(9,8),(36,6)]:
    m = build_coil_matrix(Q, P, 3, layers=2)
    assert np.all(m != 0), f"zero conductor Q={Q} P={P}"
m = build_coil_matrix(12, 10, 3, layers=2)
assert set(np.abs(m.flatten()).tolist()).issubset({1,2,3})
for Q, P in [(12,4),(12,10),(9,8)]:
    mat = build_coil_matrix(Q, P, 3, layers=1)
    counts = [int(np.sum(np.abs(mat[0])==k)) for k in [1,2,3]]
    assert len(set(counts)) == 1, f"unbalanced Q={Q} P={P}"
Q, P, w = 12, 4, 3
mat = build_coil_matrix(Q, P, 3, layers=2, w=w)
for i in range(Q):
    go = (i - w + Q) % Q
    assert mat[1, i] == -mat[0, go]
print("✓ build_coil_matrix")


In [ ]:
#| hide
import numpy as np, math
from emachines.winding.sos import (assign_phases, winding_factor_sos,
    check_symmetry, is_valid_combination, get_valid_coil_spans,
    build_coil_matrix, build_star_of_slots)

# ── assign_phases ─────────────────────────────────────────────────────
assert len(assign_phases(12, 4, 3, layers=1)) == 3
for ph in assign_phases(12, 4, 3, layers=2):
    assert len(ph) == 2
for ph in assign_phases(12, 4, 3, layers=1):
    assert ph[1] == []
for ph in assign_phases(12, 10, 3, layers=2):
    for layer in ph:
        for slot in layer:
            assert 1 <= abs(slot) <= 12
Q, P = 12, 4
phases = assign_phases(Q, P, 3, layers=2)
for lyr_idx in range(2):
    slots = [abs(s) for ph in phases for s in ph[lyr_idx]]
    assert sorted(slots) == list(range(1, Q+1))
print("✓ assign_phases")

# ── winding_factor_sos ─────────────────────────────────────────────────
cases_sos = [
    (12, 10, 1, 2, 0.933), (12, 8,  1, 2, 0.866), (9, 8, 1, 2, 0.945),
    (12, 14, 1, 2, 0.933), (12, 4, 3, 2, 1.000),  (24, 4, 6, 2, 0.966),
]
for Q, P, w, layers, expected in cases_sos:
    kw = winding_factor_sos(1, Q, P, m=3, layers=layers, w=w)
    assert math.isclose(kw, expected, abs_tol=0.01), f"Q={Q} P={P}: kw={kw:.4f}≠{expected}"
for Q, P in [(12,10),(12,8),(9,8),(12,14),(12,4),(36,6)]:
    kw = winding_factor_sos(1, Q, P, layers=2, w=1)
    assert kw <= 1.0 + 1e-9
kw1 = winding_factor_sos(1, 12, 10, layers=2, w=1)
kw3 = winding_factor_sos(3, 12, 10, layers=2, w=1)
assert kw3 <= kw1 + 1e-9
print("✓ winding_factor_sos")

# ── check_symmetry / is_valid / coil_spans ────────────────────────────
for Q, P, exp in [(12,4,True),(12,10,True),(9,8,True),(36,6,True),(10,4,False),(6,4,True)]:
    assert check_symmetry(Q, P) == exp, f"check_symmetry({Q},{P})"
for Q, P, exp in [(12,4,True),(12,10,True),(9,8,True),(10,4,False),(2,4,False),(12,12,False),(12,3,False)]:
    assert is_valid_combination(Q, P) == exp, f"is_valid_combination({Q},{P})"
assert get_valid_coil_spans(12, 4) == list(range(1, 7))
assert get_valid_coil_spans(12, 10) == [1, 2]
assert get_valid_coil_spans(9, 8)[0] == 1
Q, P = 36, 6
assert get_valid_coil_spans(Q, P)[-1] == Q // (P // 2)
print("✓ check_symmetry / is_valid_combination / get_valid_coil_spans")

# ── winding_factor_sos: phase symmetry ───────────────────────────────
Q, P = 12, 10
mat = build_coil_matrix(Q, P, 3, layers=2, w=1)
star = build_star_of_slots(Q, P)
kw_phases = []
for k in range(3):
    phasor, n = 0j, 0
    for lyr in range(2):
        for idx in range(Q):
            v = int(mat[lyr, idx])
            if v == k + 1:
                phasor += np.exp(1j * star[idx]); n += 1
            elif v == -(k + 1):
                phasor -= np.exp(1j * star[idx]); n += 1
    kw_phases.append(abs(phasor) / n if n > 0 else 0.0)
assert np.allclose(kw_phases, kw_phases[0], atol=1e-10), f"Phases not balanced: {kw_phases}"
print("✓ winding_factor_sos phase symmetry")

# ── winding_factor_sos: 5/6-chorded integer-slot ──────────────────────
# 24s/4p, w=5 (5/6 of full pole-pitch 6): kw1 ≈ 0.933
kw_chorded = winding_factor_sos(1, 24, 4, m=3, layers=2, w=5)
assert math.isclose(kw_chorded, 0.933, abs_tol=0.01), f"24s/4p w=5: kw={kw_chorded:.4f}≠0.933"
print("✓ winding_factor_sos chorded integer-slot")
print("\n✓ All winding tests passed")
